In [ ]:
# --- Environment Setup ---

# 1. Install high-performance geospatial libraries
#!pip install -q geemap pydeck ee-extra streamlit
#!npm install -q localtunnel

import ee
import geemap.deck as geemap
import pydeck as pdk
import streamlit as st
import pandas as pd
import numpy as np
import urllib

# GEE provided Project ID
MY_PROJECT_ID = 'belgaum-492521'

try:
    # Explicitly link the project ID here
    ee.Initialize(project=MY_PROJECT_ID)
    print(f"Successfully initialized with project: {MY_PROJECT_ID}")
except Exception as e:
    # If not authorized, trigger the interactive flow
    ee.Authenticate()
    ee.Initialize(project=MY_PROJECT_ID)

print("Environment Ready")

⠙⠹⠸⠼⠴⠦
up to date, audited 23 packages in 890ms
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦
2 vulnerabilities (1 high, 1 critical)

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠦Successfully initialized with project: belgaum-492521
Environment Ready


In [ ]:
%%capture
'''
import ee
import geemap.deck as geemap
import pydeck as pdk
'''

# 1. CLOUD FETCH (Belagavi Core)
belgaum_coords = [74.508, 15.849]
roi = ee.Geometry.Point(belgaum_coords).buffer(8000)
buildings = ee.FeatureCollection("GOOGLE/Research/open-buildings/v3/polygons")
local_buildings = buildings.filterBounds(roi).filter(ee.Filter.gte('confidence', 0.75))

def add_props(feature):
    total_area = ee.Number(feature.get('area_in_meters'))
    usable_area = total_area.multiply(0.6)

    # 213.5 factor = 5.2 (Sun) * 365 (Days) * 0.15 (Eff) * 0.75 (PR)
    yield_mwh = usable_area.multiply(213.5).divide(1000)

    # Rounding to 2 decimal places using GEE logic: (x * 100).round() / 100
    yield_rounded = yield_mwh.multiply(100).round().divide(100)
    area_rounded = usable_area.multiply(100).round().divide(100)

    height_val = total_area.sqrt().divide(1.5).clamp(4, 30).round()

    return feature.set({
        'usable_area_m2': area_rounded,
        'yield_mwh': yield_rounded,
        'height_m': height_val
    })

geojson_data = geemap.ee_to_geojson(local_buildings.map(add_props).limit(4000))

# 2. 3D DESIGN
layer = pdk.Layer(
    "GeoJsonLayer",
    geojson_data,
    opacity=0.9,
    extruded=True,
    pickable=True,
    get_elevation="properties.height_m * 4",
    get_fill_color="[255, (properties.yield_mwh * 15), 0, 200]", # Scaled for MWh visibility
)

view_state = pdk.ViewState(latitude=15.849, longitude=74.508, zoom=15, pitch=60, bearing=0)

r = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    map_provider="carto",
    map_style="light",
    tooltip={
        "html": "<b>Usable Area:</b> {usable_area_m2} m²<br/>"
                "<b>Annual Yield:</b> {yield_mwh} MWh/year<br/>"
                "<b>Est. Height:</b> {height_m} m"
    }
)

r.to_html("Potential Buildings.html")

# 3. INTERFACE INJECTION
script_and_legend = """
<script>
const nArrow = document.createElement('div');
nArrow.id = 'n-arrow';
nArrow.innerHTML = '<div style="font-size: 22px; font-weight: bold; color: #ffd700;">N</div><div>↑</div>';
nArrow.style = "position: absolute; top: 20px; right: 20px; z-index: 1000; color: #333; text-align: center;";
document.body.appendChild(nArrow);

window.deck.setProps({
    onViewStateChange: ({viewState}) => {
        document.getElementById('n-arrow').style.transform = `rotate(${-viewState.bearing}deg)`;
    }
});
</script>

<div style="position: absolute; bottom: 30px; left: 20px; z-index: 1000;
            background: rgba(255,255,255,0.9); color: #333; padding: 15px;
            border-radius: 8px; font-family: sans-serif; border: 1px solid #ccc; box-shadow: 0 2px 10px rgba(0,0,0,0.1);">
    <h3 style="margin: 0; color: #d4af37;">Project Sun-Drive</h3>
    <p style="font-size: 12px; margin: 5px 0 10px 0; font-weight: bold;">District: Belagavi</p>

    <div style="font-size: 11px; margin-bottom: 5px;">Annual Solar Potential (<b>MWh</b>)</div>
    <div style="background: linear-gradient(to right, #ff0000, #ffff00, #ffd700); height: 12px; width: 220px;"></div>
    <div style="display: flex; justify-content: space-between; font-size: 10px; margin-top: 5px;">
        <span>Lower Potential</span><span>Higher Potential</span>
    </div>

    <hr style="border: 0.5px solid #eee; margin: 12px 0;">
    <div style="font-size: 10px; color: #666;">
        <b>Logic:</b> 50% Net Usable Area<br>
        <b>Unit:</b> Megawatt-hours (MWh)/Year
    </div>
</div>
"""

with open("Potential Buildings.html", "a") as f: f.write(script_and_legend)

print("Stage 1 Finalized")

In [53]:
# Interactive Economics Calculator

import json

# --- BELAGAVI METEOROLOGICAL DATA ---
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
WEIGHTS = [1.15, 1.10, 1.25, 1.15, 1.05, 0.45, 0.35, 0.40, 0.75, 0.95, 1.05, 1.10]

html_template = f"""
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Roboto:wght@300;400;700&display=swap" rel="stylesheet">
    <style>
        body {{ background-color: #1e1e1e; font-family: 'Roboto', sans-serif; color: white; margin: 0; padding: 20px; }}

        /* THE COLAB BLACK CARD */
        .header-card {{
            background: #121212;
            padding: 30px;
            border-radius: 15px;
            margin-bottom: 20px;
            box-shadow: 0 4px 15px rgba(0,0,0,0.5);
        }}
        .header-title {{ color: #FFD700; margin: 0; font-size: 24px; }}
        .kpi-row {{ display: flex; justify-content: space-around; margin-top: 20px; text-align: center; }}
        .kpi-box b {{ color: #aaa; font-size: 14px; text-transform: uppercase; }}
        .kpi-val {{ display: block; font-size: 22px; font-weight: bold; margin-top: 5px; }}
        .green-text {{ color: #4CAF50; }}
        .gold-text {{ color: #FFD700; }}

        /* SLIDERS SECTION */
        .controls {{ display: flex; gap: 40px; background: #333; padding: 15px; border-radius: 10px; margin-bottom: 20px; align-items: center; }}
        .control-group {{ display: flex; align-items: center; gap: 15px; flex: 1; }}
        input[type=range] {{ flex: 1; cursor: pointer; }}

        /* PLOT GRID */
        .plot-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 15px; background: white; padding: 10px; border-radius: 5px; }}
        .footer-note {{ font-size: 11px; color: #888; margin-top: 15px; }}
    </style>
</head>
<body>

<div class="header-card">
    <h2 class="header-title">Project Sun-Drive: <span id="sysKW">3.0</span>KW Audit</h2>
    <div class="kpi-row">
        <div class="kpi-box"><b>Net Investment</b><span id="invVal" class="kpi-val green-text">₹117,000</span></div>
        <div class="kpi-box"><b>Annual Revenue</b><span id="revVal" class="kpi-val">₹23,328</span></div>
        <div class="kpi-box"><b>Payback Period</b><span id="payVal" class="kpi-val gold-text">5.0 Years</span></div>
    </div>
    <p class="footer-note">*Calculation includes 60% usable area logic and PM-Surya Ghar subsidy capping.</p>
</div>

<div class="controls">
    <div class="control-group">
        <label>Total SqFt: <b id="sqftDisp">1200</b></label>
        <input type="range" id="areaSlider" min="500" max="4000" step="50" value="1200" oninput="update()">
    </div>
    <div class="control-group">
        <label>Consumption Strategy:</label>
        <select id="stratSelect" onchange="update()" style="padding:5px; border-radius:5px;">
            <option value="50">50% (High Export)</option>
            <option value="75" selected>75% (Balanced)</option>
            <option value="100">100% (Full Save)</option>
        </select>
    </div>
</div>

<div class="plot-grid">
    <div id="plotLeft"></div>
    <div id="plotRight"></div>
</div>

<script>
const MONTHS = {json.dumps(MONTHS)};
const WEIGHTS = {json.dumps(WEIGHTS)};

function update() {{
    const area = parseFloat(document.getElementById('areaSlider').value);
    const strat = parseFloat(document.getElementById('stratSelect').value) / 100;
    document.getElementById('sqftDisp').innerText = area;

    // --- PHYSICS & MATH ---
    const usableArea = area * 0.60;
    const kw = Math.max(1.0, Math.round((usableArea / 100) * 10) / 10);
    const annUnits = kw * 1400;
    const mUnits = WEIGHTS.map(w => (annUnits / 12) * w);
    const mRev = mUnits.map(u => u * (strat * 7.5 + (1-strat) * 2.3));

    // Subsidy Logic
    let subsidy = kw <= 2 ? kw * 30000 : (kw <= 3 ? 60000 + (kw-2)*18000 : 78000);
    const netInv = (kw * 65000) - subsidy;
    const totalAnnRev = mRev.reduce((a, b) => a + b, 0);
    const payback = netInv / totalAnnRev;

    // Update Header
    document.getElementById('sysKW').innerText = kw;
    document.getElementById('invVal').innerText = "INR " + Math.round(netInv).toLocaleString();
    document.getElementById('revVal').innerText = "INR " + Math.round(totalAnnRev).toLocaleString();
    document.getElementById('payVal').innerText = payback.toFixed(1) + " Years";

    // --- PLOTS ---
    const layoutBase = {{
        margin: {{t:40, b:40, l:50, r:50}},
        paper_bgcolor: 'white',
        plot_bgcolor: 'white',
        font: {{ family: 'Roboto', size: 10 }}
    }};

    // Left Plot: Bar + Line
    const trace1 = {{ x: MONTHS, y: mUnits, type: 'bar', name: 'Units (kWh)', marker: {{color: '#fbc02d', opacity: 0.6}} }};
    const trace2 = {{ x: MONTHS, y: mRev, type: 'scatter', mode: 'lines+markers', name: 'Revenue (INR)', yaxis: 'y2', line: {{color: '#2e7d32'}} }};

    Plotly.newPlot('plotLeft', [trace1, trace2], {{
        ...layoutBase,
        title: `Monthly Energy Harvest (${{kw}} kW System)`,
        yaxis: {{ title: 'Units (kWh)' }},
        yaxis2: {{ title: 'Revenue (INR)', overlaying: 'y', side: 'right' }},
        legend: {{ orientation: 'h', y: -0.2 }}
    }});

    // Right Plot: 20-Year ROI
    let years = Array.from({{length: 21}}, (_, i) => i);
    let cumulative = [-netInv];
    for(let i=1; i<=20; i++) {{
        cumulative.push(cumulative[i-1] + (totalAnnRev * Math.pow(1.03, i) * Math.pow(0.995, i)));
    }}

    Plotly.newPlot('plotRight', [{{
        x: years, y: cumulative, fill: 'tozeroy', type: 'scatter',
        line: {{color: 'darkgreen'}}, fillcolor: 'rgba(0, 128, 0, 0.2)'
    }}], {{
        ...layoutBase,
        title: '20-Year Cumulative Cash Flow',
        xaxis: {{ title: 'Years' }},
        yaxis: {{ title: 'Net Profit/Loss (INR)' }},
        shapes: [{{ type: 'line', x0: 0, y0: 0, x1: 20, y1: 0, line: {{color: 'red', dash: 'dash', width: 1}} }}]
    }});
}}

// Initial call
update();
</script>

</body>
</html>
"""

with open("SunDrive_Interactive.html", "w") as f:
    f.write(html_template)

In [52]:
import json

# --- BELAGAVI DATA ---
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
WEIGHTS = [1.15, 1.10, 1.25, 1.15, 1.05, 0.45, 0.35, 0.40, 0.75, 0.95, 1.05, 1.10]

html_template = f"""
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap" rel="stylesheet">
    <style>
        :root {{
            --sky-bg: #E0F2FE;
            --navy-card: #0F172A;
            --accent-gold: #FBBF24;
            --success-green: #4ADE80;
            --co2-teal: #0D9488;
            --text-slate: #334155;
        }}
        body {{ background-color: var(--sky-bg); font-family: 'Inter', sans-serif; margin: 0; display: flex; height: 100vh; overflow: hidden; }}

        /* SIDEBAR (30%) */
        .sidebar {{
            width: 30%; background: white; border-right: 1px solid #BAE6FD;
            padding: 30px; display: flex; flex-direction: column; box-sizing: border-box;
        }}

        /* MAIN STAGE (70%) */
        .main-stage {{
            width: 70%; padding: 25px; display: flex; flex-direction: column;
            gap: 20px; box-sizing: border-box;
        }}

        .header-card {{ background: var(--navy-card); padding: 30px; border-radius: 20px; color: white; }}
        .title {{ color: var(--accent-gold); font-size: 1.8rem; font-weight: 700; margin-bottom: 20px; }}

        .kpi-row {{ display: flex; justify-content: space-between; text-align: center; }}
        .kpi-label {{ font-size: 0.85rem; color: #94A3B8; text-transform: uppercase; font-weight: 600; }}
        .kpi-val {{ display: block; font-size: 1.8rem; font-weight: 700; margin-top: 5px; }}

        .plot-box {{ background: white; border-radius: 15px; flex: 1; padding: 15px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); }}

        /* SIDEBAR COMPONENTS */
        .control-section {{ margin-bottom: 30px; }}
        .env-report-box {{
            background: #F0FDFA; border: 1.5px solid #CCFBF1; border-radius: 12px;
            padding: 20px; margin-top: auto;
        }}

        label {{ font-weight: 700; color: var(--text-slate); text-transform: uppercase; font-size: 0.85rem; display: block; margin-bottom: 10px; }}
        input[type=range] {{ width: 100%; accent-color: #0284C7; cursor: pointer; }}
        input[type=number], select {{
            width: 100%; padding: 12px; border: 2px solid #E2E8F0; border-radius: 10px; font-size: 1rem; font-family: inherit;
        }}
    </style>
</head>
<body>

<div class="sidebar">
    <h2 style="color: #0369A1; margin-top: 0;">Planner Toolkit</h2>

    <div class="control-section">
        <label>Avg. Rooftop Area: <span id="areaDisp" style="color:#0284C7">1200</span> sq.ft</label>
        <input type="range" id="area" min="500" max="4000" step="50" value="1200" oninput="update()">
    </div>

    <div class="control-section">
        <label>Number of Houses</label>
        <input type="number" id="units" value="1" min="1" step="1" oninput="update()">
    </div>

    <div class="control-section">
        <label>Consumption Strategy</label>
        <select id="strategy" onchange="update()">
            <option value="0.75" selected>75% (Balanced)</option>
            <option value="1.00">100% (Full Save)</option>
        </select>
    </div>

    <div class="env-report-box">
        <label style="color: var(--co2-teal); font-size: 1rem;"> Climate Mitigation Impact</label>
        <div style="margin-top:15px;">
            <span style="font-size: 0.8rem; color: #555;">TOTAL CO2 REDUCTION:</span>
            <span id="totalCO2" style="display:block; font-size: 1.8rem; font-weight: 800; color: var(--co2-teal);">0 kg</span>
        </div>
        <div style="margin-top:10px; font-size: 0.85rem; color: #64748B;">
            Equivalent to planting <b id="trees" style="color: #0D9488;">0</b> mature trees per year.
        </div>
    </div>
</div>

<div class="main-stage">
    <div class="header-card">
        <div class="title">Project Sun-Drive: <span id="sysKW">0</span> kW Community Audit</div>
        <div class="kpi-row">
            <div><span class="kpi-label">Investment (Net)</span><span id="invVal" class="kpi-val" style="color:var(--success-green);">INR 0</span></div>
            <div><span class="kpi-label">Annual Savings</span><span id="revVal" class="kpi-val">INR 0</span></div>
            <div><span class="kpi-label">Avg. Payback</span><span id="payVal" class="kpi-val" style="color:var(--accent-gold);">0 Years</span></div>
        </div>
    </div>

    <div id="harvestPlot" class="plot-box"></div>
    <div id="cashflowPlot" class="plot-box"></div>
</div>

<script>
const MONTHS = {json.dumps(MONTHS)};
const WEIGHTS = {json.dumps(WEIGHTS)};

function update() {{
    const area = parseFloat(document.getElementById('area').value);
    const units = parseInt(document.getElementById('units').value) || 1;
    const strategy = parseFloat(document.getElementById('strategy').value);
    document.getElementById('areaDisp').innerText = area;

    // --- COMMUNITY LOGIC ---
    const kwPerHouse = Math.max(1, Math.floor((area * 0.6) / 100));
    const annkWhSingle = kwPerHouse * 1400;

    // Financials (Single Unit for Header)
    const netInvSingle = (kwPerHouse * 65000) - (kwPerHouse >= 3 ? 78000 : kwPerHouse * 30000);
    const annRevSingle = annkWhSingle * (strategy * 7.5 + (1 - strategy) * 2.3);

    // Total Sustainability (Multi-Unit)
    const totalAnnkWh = annkWhSingle * units;
    const totalCO2Val = Math.floor(totalAnnkWh * 0.8); // 0.8kg per kWh
    const treeCount = Math.floor(totalCO2Val / 21); // Avg tree absorbs 21kg/year

    // Update Header & Sidebar
    document.getElementById('sysKW').innerText = kwPerHouse;
    document.getElementById('invVal').innerText = "INR " + Math.round(netInvSingle).toLocaleString();
    document.getElementById('revVal').innerText = "INR " + Math.round(annRevSingle).toLocaleString();
    document.getElementById('payVal').innerText = (netInvSingle / annRevSingle).toFixed(1) + " Years";

    document.getElementById('totalCO2').innerText = totalCO2Val.toLocaleString() + " kg";
    document.getElementById('trees').innerText = treeCount.toLocaleString();

    // --- PLOTS ---
    const mkWh = WEIGHTS.map(w => Math.floor((annkWhSingle / 12) * w));
    const mRev = mkWh.map(u => Math.floor(u * (strategy * 7.5 + (1 - strategy) * 2.3)));

    Plotly.newPlot('harvestPlot', [
        {{ x: MONTHS, y: mkWh, type: 'bar', name: 'Units (kWh)', marker: {{color: '#FBBF24'}} }},
        {{ x: MONTHS, y: mRev, type: 'scatter', name: 'Revenue (INR)', yaxis: 'y2', line: {{color: '#059669', width: 3}} }}
    ], {{
        title: {{ text: '<b>Single House Monthly Performance</b>', font: {{size: 16}} }},
        yaxis: {{ title: 'Energy (kWh)' }},
        yaxis2: {{ title: 'Revenue (INR)', overlaying: 'y', side: 'right', showgrid: false }},
        margin: {{t:50, b:40, l:60, r:60}},
        legend: {{ orientation: 'h', y: -0.2 }}
    }}, {{responsive: true}});

    let years = Array.from({{length: 21}}, (_, i) => i);
    let cf = [-netInvSingle];
    for(let i=1; i<=20; i++) cf.push(Math.floor(cf[i-1] + (annRevSingle * Math.pow(1.02, i))));

    Plotly.newPlot('cashflowPlot', [{{
        x: years, y: cf, fill: 'tozeroy', line: {{color: '#0369A1'}}, fillcolor: 'rgba(3, 105, 161, 0.1)'
    }}], {{
        title: {{ text: '<b>20-Year Individual ROI Projection</b>', font: {{size: 16}} }},
        yaxis: {{ title: 'Cumulative Balance (INR)' }},
        margin: {{t:50, b:40, l:70, r:40}}
    }}, {{responsive: true}});
}}

update();
</script>
</body>
</html>
"""

with open("SunDrive_Final.html", "w", encoding="utf-8") as f:
    f.write(html_template)